# 자연어 처리 7일차 (1) — SentenceTransformer 로 문장 임베딩 · 문장 유사도

수업 필기 `nlp_실습11` 을 **실행 가능하도록 정리 · 개선한 노트**입니다.
(교재 `자연어처리` 교재v1 **p.115~117** — SentenceTransformer · 문장 임베딩 만드는 방법(평균) · 용도 · 문장 유사도 측정 실습)

```
문장 → BERT 계열 모델 → 토큰마다 벡터 (길이, 768) → 평균(mean pooling) → 문장 벡터 (768,) → 코사인 유사도
```

- 이 문장 벡터로 만든 **영화 추천**은 [`movie_recommendation_sbert.ipynb`](./movie_recommendation_sbert.ipynb), **챗봇**(필기 `nlp_실습12`)은 [`chatbot_similarity.ipynb`](./chatbot_similarity.ipynb) 에 있습니다.
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다. **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 있습니다.

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `sentence-transformers` | `SentenceTransformer` · `encode()` · `similarity()` | Colab 기본 포함 · 로컬은 `pip install sentence-transformers` |
| `transformers` · `torch` | 평균 풀링 손계산 | 위와 함께 설치됨 |

**모델** (첫 실행 때 Hugging Face 에서 자동 다운로드 → **인터넷 필요**)

| 모델 | 언어 | 벡터 크기 | 크기 |
|---|---|---|---|
| `sentence-transformers/all-MiniLM-L12-v2` | 영어 | 384 | 133MB |
| `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` | 50여 개 언어 | 384 | 471MB |
| `ddobokki/klue-roberta-base-nli-sts` | 한국어 | 768 | 442MB |
| `intfloat/multilingual-e5-large-instruct` | 다국어 | 1024 | 1.1GB |

**실행 순서** : 맨 위부터 **[런타임] → [모두 실행]**. 문장 몇 개만 임베딩하므로 GPU 없이 CPU 로 1~2분(다운로드 제외).

In [1]:
# ① 설치 (로컬에서 처음 한 번 — Colab 은 기본 포함)
# !pip install -q sentence-transformers

In [2]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'      # 다운로드 진행 막대 숨기기
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
from huggingface_hub.utils import logging as hub_logging
hub_logging.set_verbosity_error()                     # 'unauthenticated requests' 안내 숨기기 (공개 모델은 로그인 없이 받아진다)

In [3]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

## 1. 영어 문장 유사도

In [4]:
en_sentences = ['What should I do to be a great scientist?',
                'How can I be a good scientist?',
                'Where can I buy cheap flight tickets?']

model_eng = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
en_embeddings = model_eng.encode(en_sentences)                  # (문장 수, 384) numpy 배열
print('임베딩 모양 :', en_embeddings.shape, '| 앞 5칸 :', en_embeddings[0, :5].round(3))
print('코사인 유사도 :\n', model_eng.similarity(en_embeddings, en_embeddings))

임베딩 모양 : (3, 384) | 앞 5칸 : [ 0.022  0.085  0.079 -0.007 -0.015]
코사인 유사도 :
 tensor([[1.0000, 0.8955, 0.0707],
        [0.8955, 1.0000, 0.0977],
        [0.0707, 0.0977, 1.0000]])


같은 뜻(과학자가 되는 법) 두 문장은 0.90 근처, 엉뚱한 문장(항공권)과는 낮다. `model.similarity()` 의 기본값은 **코사인 유사도** (`model.similarity_fn_name` 으로 확인).

## 2. 문장 벡터는 어떻게 만드나 — 평균 풀링 (교재 p.115)

In [5]:
print(model_eng)                                                # Transformer → Pooling → Normalize
print('\n풀링 :', model_eng[1], '| 유사도 함수 :', model_eng.similarity_fn_name, '| 최대 길이 :', model_eng.max_seq_length)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

풀링 : Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True}) | 유사도 함수 : cosine | 최대 길이 : 128


In [6]:
# 손으로 : 토큰 벡터들의 평균 (패딩 칸은 빼고) → 길이 1 로 정규화
tok, bert = model_eng.tokenizer, model_eng[0].auto_model
x = tok(en_sentences[0], return_tensors='pt')
with torch.no_grad():
    token_vecs = bert(**x).last_hidden_state[0]                # (토큰 수, 384)
mask = x['attention_mask'][0].unsqueeze(1)
sent_vec = (token_vecs * mask).sum(0) / mask.sum()             # 평균 풀링
sent_vec = sent_vec / sent_vec.norm()                          # Normalize 층

print('토큰 :', tok.convert_ids_to_tokens(x['input_ids'][0]))
print('토큰 벡터', tuple(token_vecs.shape), '→ 평균 →', tuple(sent_vec.shape))
print('encode() 결과와 같은가 ?', np.allclose(sent_vec.numpy(), en_embeddings[0], atol=1e-5))

토큰 : ['[CLS]', 'what', 'should', 'i', 'do', 'to', 'be', 'a', 'great', 'scientist', '?', '[SEP]']
토큰 벡터 (12, 384) → 평균 → (384,)
encode() 결과와 같은가 ? True


> **📝 고친 부분 (교재 p.115)** 'BERT 의 모든 출력 벡터에 대해 평균' 은 맞지만, **`[PAD]` 칸은 빼고**(attention mask) 평균을 낸다. 그리고 SentenceTransformer 모델은 그냥 BERT 가 아니라 **문장 쌍 데이터(NLI · STS · 질문-답 쌍)로 '같은 뜻이면 가깝게' 추가 학습**한 모델이다 — 그냥 BERT 출력의 평균은 문장 유사도를 잘 재지 못한다(아래 §5). 그림의 `(Max)` 는 칸마다 최댓값을 고르는 max pooling, `[CLS]` 벡터만 쓰는 방법도 있다.

## 3. 한국어 문장 유사도 — 다국어 모델 vs 한국어 모델

In [7]:
ko_sentences = ['직원이 무단 퇴사를 했는데 손해 배상 청구할 수 있나요?',
                '무단 퇴사한 직원에 대한 손해 배상 청구가 가능한가요?',
                '오늘도 날씨가 매우 덥고 소나기가 내리는 등 동남아 날씨같아요.']
labels = ['퇴사 질문 1', '퇴사 질문 2', '날씨']

model_multi = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
model_kor = SentenceTransformer('ddobokki/klue-roberta-base-nli-sts')
for name, m in [('다국어 MiniLM', model_multi), ('한국어 KLUE-RoBERTa', model_kor)]:
    emb = m.encode(ko_sentences)
    sim = m.similarity(emb, emb)
    print(f'[{name}] 벡터 {emb.shape[1]}칸 · 같은 뜻 {sim[0, 1]:.4f} · 무관 {sim[0, 2]:.4f}')

[다국어 MiniLM] 벡터 384칸 · 같은 뜻 0.9178 · 무관 -0.1399
[한국어 KLUE-RoBERTa] 벡터 768칸 · 같은 뜻 0.9606 · 무관 -0.0605


원본과 같이 두 모델 모두 같은 뜻 0.92 · 0.96, 무관한 문장은 0 근처(음수도 나온다 — 코사인은 −1 ~ 1). 세 문장을 한 번에 `encode` 하면 비교 행렬이 한 번에 나온다.

> **📝 고친 부분 (원본 cell 9~15)** 같은 첫 문장을 두 리스트(`ko_sentences1` · `ko_sentences2`)에 넣어 **두 번 임베딩** · `similarity` 도 2×2 행렬을 두 번 → 세 문장을 한 번에 `encode` 하고 필요한 칸만 본다. 오타 `임배딩` · `호츨` · `백터` → 임베딩 · 호출 · 벡터.

## 4. 4개 언어 같은 뜻 문장 — multilingual-e5-large-instruct

In [8]:
sentences = ['What should I do to be a great scientist?',
             '훌륭한 과학자가 되려면 어떻게 해야 할까요?',
             '怎样才能成为一名优秀的科学家？',
             '¿Qué debo hacer para convertirme en un gran científico?',
             '오늘 점심 메뉴는 김치찌개입니다.']                     # 비교용 : 무관한 문장
names = ['영어', '한국어', '중국어', '스페인어', '무관(점심)']

model_e5 = SentenceTransformer('intfloat/multilingual-e5-large-instruct',
                               model_kwargs={'torch_dtype': torch.float32})   # 파일은 float16 — CPU 에서는 float32 로
print('가중치 자료형 :', next(model_e5.parameters()).dtype)
emb = model_e5.encode(sentences, normalize_embeddings=True)
sim = model_e5.similarity(emb, emb)
print('모양 :', emb.shape)
import pandas as pd
pd.DataFrame(sim.numpy().round(3), index=names, columns=names)

가중치 자료형 : torch.float32


모양 : (5, 1024)


,영어,한국어,중국어,스페인어,무관(점심)
영어,1.000,0.926,0.928,0.957,0.690
한국어,0.926,1.000,0.956,0.933,0.753
중국어,0.928,0.956,1.000,0.929,0.717
스페인어,0.957,0.933,0.929,1.000,0.718
무관(점심),0.690,0.753,0.717,0.718,1.000


- 언어가 달라도 같은 뜻이면 0.92~0.96 — 그런데 **무관한 점심 문장도 0.69~0.75** 로 꽤 높다. e5 계열은 유사도가 대부분 0.7~1.0 사이에 몰려 나오도록 학습되어 **점수의 절댓값을 다른 모델과 비교하면 안 되고**, 같은 모델 안에서 **순위**로 본다.
- e5 모델은 입력에 접두어를 붙이도록 학습됐다 — 검색이면 질문에 `Instruct: …\nQuery: …`, 문서에는 접두어 없이(`-instruct` 모델 카드). 같은 종류끼리 비교할 때는 위처럼 그대로 넣어도 된다.

> **📝 고친 부분 (원본 cell 20)** 원본 결과의 자기 자신 유사도가 `0.9995`(1 이 아님) · `dtype=torch.float16` — 이 모델은 가중치 파일이 **반정밀도(float16)** 라 그대로 불러오면 float16 으로 계산되어 생긴 반올림 오차다. `model_kwargs={'torch_dtype': torch.float32}` 로 불러오면 1.000 (CPU 는 float16 이 오히려 느리다는 안내도 사라진다). GPU 에서는 float16 이 빠르고 메모리도 절반이라 그대로 써도 된다.

## 5. 문장용으로 학습하지 않은 BERT 평균은?

In [9]:
from transformers import AutoTokenizer, AutoModel
plain_tok = AutoTokenizer.from_pretrained('klue/bert-base')
plain = AutoModel.from_pretrained('klue/bert-base')

def plain_mean(texts):
    x = plain_tok(texts, padding=True, return_tensors='pt')
    with torch.no_grad():
        h = plain(**x).last_hidden_state
    m = x['attention_mask'].unsqueeze(-1)
    v = (h * m).sum(1) / m.sum(1)
    return torch.nn.functional.normalize(v, dim=-1)

v = plain_mean(ko_sentences)
print('klue/bert-base 평균 (문장 학습 X) — 같은 뜻', round((v[0] @ v[1]).item(), 3), '· 무관', round((v[0] @ v[2]).item(), 3))
e = model_kor.encode(ko_sentences, normalize_embeddings=True)
print('klue-roberta-nli-sts (문장 학습 O) — 같은 뜻', round(float(e[0] @ e[1]), 3), '· 무관', round(float(e[0] @ e[2]), 3))

klue/bert-base 평균 (문장 학습 X) — 같은 뜻 0.905 · 무관 0.47
klue-roberta-nli-sts (문장 학습 O) — 같은 뜻 0.961 · 무관 -0.06


문장 쌍으로 추가 학습하지 않은 BERT 평균은 **무관한 문장끼리도 유사도가 높게** 나와 구분이 약하다. 그래서 '문장 임베딩 = BERT 평균' 이 아니라 **문장용으로 학습된 SentenceTransformer 모델**을 쓴다.

## 정리

| 할 일 | 코드 |
|---|---|
| 모델 | `SentenceTransformer('모델 이름')` |
| 문장 벡터 | `model.encode([문장, …])` → (문장 수, 차원) |
| 유사도 행렬 | `model.similarity(a, b)` (기본 코사인) |
| 만드는 방법 | 토큰 벡터 평균(mean pooling, 패딩 제외) → 정규화 |
| 주의 | 점수 범위는 모델마다 다르다 — 같은 모델 안에서 순위로 비교 |